In [ ]:
import pandas as pd
import glob

# Find all labelled files
files = glob.glob('data/raw/**/*.labeled', recursive=True)
print(f"Found {len(files)} labelled files:")
for f in files:
    print(f"  {f}")

# Load one scenario
df = pd.read_csv(files[0], sep='\t', comment='#',
                 low_memory=False, header=None)

# Extract column names from the #fields line
with open(files[0]) as fh:
    for line in fh:
        if line.startswith('#fields'):
            cols = line.strip().split('\t')[1:]
            break
df.columns = cols

print(f"\n--- STRUCTURE ---")
print(f"Rows: {len(df):,}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")

print(f"\n--- ATTACK LABELS ---")
print(df['label'].value_counts())

print(f"\n--- DETAILED LABELS ---")
print(df['detailed-label'].value_counts())

print(f"\n--- DATA TYPES ---")
print(df.dtypes)

print(f"\n--- MISSING VALUES (Zeek dashes) ---")
dash_counts = (df == '-').sum()
print(dash_counts[dash_counts > 0])

print(f"\n--- SAMPLE ROWS ---")
print(df.head(3).to_string())

In [ ]:
import pandas as pd
import glob
import os

# ── Path to your extracted IoT-23 dataset ──────────────────────────
DATA_ROOT = r"C:\Users\Asus\OneDrive\Desktop\MSc Cybersecurity -NTU\Major Project\Dataset\iot_23_datasets_small"

# Find all labelled log files
files = glob.glob(DATA_ROOT + '/**/*.labeled', recursive=True)
print(f"Found {len(files)} labelled files:\n")
for f in files:
    print(f"  {os.path.basename(f)}  ←  {os.path.dirname(f).split(os.sep)[-1]}")

# ── Load first scenario ────────────────────────────────────────────
df = pd.read_csv(files[0], sep='\t', comment='#',
                 low_memory=False, header=None)

# Extract column names from the #fields header line
with open(files[0]) as fh:
    for line in fh:
        if line.startswith('#fields'):
            cols = line.strip().split('\t')[1:]
            break
df.columns = cols

# ── Answer supervisor's 6 questions ───────────────────────────────
print(f"\n{'='*50}")
print(f"1. FORMAT: Tab-separated Zeek/Bro connection logs")
print(f"2. ORIENTATION: Rows = individual network flows | Columns = traffic features")
print(f"3. CAPTURE SOURCE: Network traffic from infected Raspberry Pi devices")
print(f"   (Stratosphere Lab, Czech Technical University — NOT physical sensors)")
print(f"{'='*50}")

print(f"\n4. DIMENSIONS: {len(df):,} rows  x  {len(df.columns)} columns")
print(f"   Columns: {list(df.columns)}")

print(f"\n5. ATTACK INDICATORS (target variables):")
print(f"   'label' column:\n{df['label'].value_counts().to_string()}")
print(f"\n   'detailed-label' column:\n{df['detailed-label'].value_counts().to_string()}")

print(f"\n6. MISSING VALUES (Zeek uses '-' as placeholder, not NaN):")
dash_counts = (df == '-').sum()
dash_counts = dash_counts[dash_counts > 0]
print(dash_counts.to_string() if len(dash_counts) > 0 else "  None found")

print(f"\n{'='*50}")
print("SAMPLE ROWS:")
print(df.head(3).to_string())

In [ ]:
import pandas as pd
import glob
import os

# ── Path to your extracted IoT-23 dataset ──────────────────────────
DATA_ROOT = r"C:\Users\Asus\OneDrive\Desktop\MSc Cybersecurity -NTU\Major Project\Dataset\iot_23_datasets_small"

# Find all labelled log files
files = glob.glob(DATA_ROOT + '/**/*.labeled', recursive=True)
print(f"Found {len(files)} labelled files")
print(f"Loading: {files[0]}\n")

# ── Read the file and extract header ──────────────────────────────
# First, print raw header lines so we can see exactly what's there
print("=== RAW HEADER LINES ===")
with open(files[0]) as fh:
    for line in fh:
        if line.startswith('#'):
            print(repr(line.strip()))  # repr() shows hidden characters
        else:
            break  # stop at first data line

# ── Load the file ─────────────────────────────────────────────────
df = pd.read_csv(files[0], sep='\t', comment='#',
                 low_memory=False, header=None)

# Extract column names from the #fields line
with open(files[0]) as fh:
    for line in fh:
        if line.startswith('#fields'):
            cols = line.strip().split('\t')[1:]
            break

# Strip any hidden whitespace from column names
cols = [c.strip() for c in cols]
df.columns = cols

print(f"\n=== ACTUAL COLUMN NAMES ({len(cols)} total) ===")
for i, c in enumerate(cols):
    print(f"  [{i}] '{c}'")

print(f"\n=== LAST 3 COLUMNS (attack indicators are usually here) ===")
print(df.iloc[:, -3:].head(5))

In [ ]:
import pandas as pd
import glob
import os

# ── Path ──────────────────────────────────────────────────────────
DATA_ROOT = r"C:\Users\Asus\OneDrive\Desktop\MSc Cybersecurity -NTU\Major Project\Dataset\iot_23_datasets_small"

files = glob.glob(DATA_ROOT + '/**/*.labeled', recursive=True)
print(f"Found {len(files)} labelled files")

# ── Load file and extract column names ────────────────────────────
df = pd.read_csv(files[0], sep='\t', comment='#',
                 low_memory=False, header=None)

with open(files[0]) as fh:
    for line in fh:
        if line.startswith('#fields'):
            cols = line.strip().split('\t')[1:]
            break

cols = [c.strip() for c in cols]
df.columns = cols

# ── FIX: split the merged last column into 3 separate columns ─────
# The last column contains 'tunnel_parents   label   detailed-label'
# because the file uses spaces instead of tabs for these 3 fields
last_col = df.columns[-1]  # 'tunnel_parents   label   detailed-label'

split_cols = df[last_col].str.strip().str.split(r'\s{2,}', expand=True, n=2)
split_cols.columns = ['tunnel_parents', 'label', 'detailed-label']

df = df.drop(columns=[last_col])
df = pd.concat([df, split_cols], axis=1)

print(f"Fixed columns ({len(df.columns)} total): {list(df.columns)}\n")

# ══════════════════════════════════════════════════════════════════
# SUPERVISOR'S 6 QUESTIONS
# ══════════════════════════════════════════════════════════════════

print("=" * 60)
print("1. FORMAT")
print("   Tab-separated Zeek/Bro network connection logs (.labeled)")

print("\n2. ORIENTATION")
print(f"   Rows = individual network flows:  {len(df):,}")
print(f"   Columns = traffic features:       {len(df.columns)}")

print("\n3. CAPTURE SOURCE")
print("   Network traffic captured from infected Raspberry Pi devices")
print("   in a controlled lab (Stratosphere Lab, Czech Tech University)")
print("   NOT from physical sensors — pure network-layer captures")
print(f"   File loaded: {os.path.basename(os.path.dirname(files[0]))}")

print("\n4. NUMBER OF FEATURES")
print(f"   {len(df.columns)} columns total:")
for i, c in enumerate(df.columns):
    print(f"     [{i:02d}] {c}")

print("\n5. ATTACK INDICATORS (target variables)")
print("\n   'label' — primary attack/benign flag:")
print(df['label'].value_counts().to_string())
print("\n   'detailed-label' — attack sub-type:")
print(df['detailed-label'].value_counts().to_string())

print("\n6. MISSING VALUES")
print("   Zeek uses '-' as placeholder (not NaN) — counting dashes:")
dash_counts = (df == '-').sum()
dash_counts = dash_counts[dash_counts > 0].sort_values(ascending=False)
print(dash_counts.to_string())

print("\n" + "=" * 60)
print("SAMPLE ROWS (first 3):")
print(df[['ts','proto','duration','orig_bytes','resp_bytes',
          'conn_state','label','detailed-label']].head(3).to_string())

In [ ]:
# ── Load ALL 23 files and get the complete picture ─────────────────
all_dfs = []

for f in files:
    try:
        tmp = pd.read_csv(f, sep='\t', comment='#',
                          low_memory=False, header=None)
        with open(f) as fh:
            for line in fh:
                if line.startswith('#fields'):
                    cols = [c.strip() for c in line.strip().split('\t')[1:]]
                    break
        tmp.columns = cols

        # Fix merged last column
        last_col = tmp.columns[-1]
        if '   ' in last_col:
            split_cols = tmp[last_col].str.strip().str.split(r'\s{2,}', expand=True, n=2)
            split_cols.columns = ['tunnel_parents', 'label', 'detailed-label']
            tmp = tmp.drop(columns=[last_col])
            tmp = pd.concat([tmp, split_cols], axis=1)

        tmp['source_file'] = os.path.basename(os.path.dirname(os.path.dirname(f)))
        all_dfs.append(tmp)
    except Exception as e:
        print(f"Skipped {f}: {e}")

full_df = pd.concat(all_dfs, ignore_index=True)

print("=" * 60)
print(f"FULL DATASET — ALL 23 FILES COMBINED")
print(f"Total rows:    {len(full_df):,}")
print(f"Total columns: {len(full_df.columns)}")

print(f"\nCLASS DISTRIBUTION ('label'):")
label_counts = full_df['label'].value_counts()
print(label_counts.to_string())
pct_attack = (full_df['label'] != 'benign').sum() / len(full_df) * 100
print(f"\nAttack traffic: {pct_attack:.1f}%")
print(f"Benign traffic: {100-pct_attack:.1f}%")

print(f"\nDETAILED ATTACK TYPES ('detailed-label'):")
print(full_df['detailed-label'].value_counts().to_string())

print(f"\nSCENARIOS (source files):")
print(full_df['source_file'].value_counts().to_string())

print(f"\nFEATURES TO DROP (all-dash columns — no useful signal):")
dash_counts = (full_df == '-').sum()
all_dash = dash_counts[dash_counts == len(full_df)]
print(list(all_dash.index))

print(f"\nCLASS IMBALANCE RATIO:")
benign_n = (full_df['label'] == 'benign').sum()
attack_n = (full_df['label'] != 'benign').sum()
if attack_n > 0:
    print(f"  Benign : Attack = {benign_n:,} : {attack_n:,}  ({benign_n/attack_n:.1f}:1)")